<h1>Detecting planets with the _Kepler_ space mission</h1>

During its original mission (2009-2013), _Kepler_ measured the brightness of about 150,000 stars every ~30 mins for four years. One of the main scieince goals was to look for the small, periodic dips caused by transiting planets. This notebook works through some of the main steps to take a Kepler light curve and detect and measure a transiting planet's properties:

1. **Raw and corrected light curves**: what Kepler measured, and how the mission's pipeline removed instrumental effects;
2. **Flattening**: removing the star's own slow brightness changes;
3. **Searching for transits** with the Box Least Squares (BLS) periodogram;
4. **Cleaning**: masking the transits found, looking for further signals, and removing outliers;
5. **Modelling the transit** by hand, with sliders;
6. **Fitting a transit model** to the data.

Each step uses the result of the previous one. When you change something in one step, all later steps are recalculated (this can take a few seconds).

Choose a star below. The three pre-selected stars each host a known planet (Kepler-7 b and Kepler-8 b are hot Jupiters; Kepler-10 b is a hot, rocky planet only 1.5 times the size of the Earth). You can also download another star with a known Kepler planet (named Kepler-1, Kepler-2, etc.; the notebook focuses on 'b' planets, i.e. the first detected planet in that system, if it is a multi-planet system) from the mission archive (MAST). Choosing a different star needs a working internet connection and takes about a minute.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from html import escape
from IPython.display import display, HTML
import ipywidgets as widgets

import kepler_tools as kt

# Figures shown with widgets live in their own output area (widgets.Output).
# show_in() replaces what an output area shows, sending the new content to the browser once.
# (Using 'with out: display(...)' instead sends it back and forth, and at the top level of a
# cell it can make Voila, the app view, hang.)
def show_in(out, obj):
    with out.hold_sync():
        out.outputs = ()
        out.append_display_data(obj)

def busy(out, text='Working, please wait...'):
    show_in(out, HTML(f'<p><i>{text}</i></p>'))

# Try and make plots look nice
plt.rcParams["font.family"] = 'serif'
plt.rcParams["mathtext.fontset"] = 'stix'
plt.rcParams["font.serif"] = ['Palatino', 'TeX Gyre Pagella', 'DejaVu Serif']   # first available is used
plt.rcParams['pdf.fonttype'] = 42

S = {}                                 # results of each step, passed on to the next step
targets = kt.bundled_targets()
style = {'description_width': 'initial'}
OTHER = 'Other Kepler star (download)'

w_target = widgets.Dropdown(options=list(targets.index) + [OTHER], value='Kepler-10', description='Star:')
w_number = widgets.BoundedIntText(value=1, min=1, max=2000, description='Kepler-', style=style,
                                  layout=widgets.Layout(width='160px'))
w_download = widgets.Button(description='Download')
other_box = widgets.HBox([w_number, w_download], layout=widgets.Layout(display='none'))
out_load = widgets.Output()
display(w_target, other_box, out_load)

<h2>1. Raw and instrument-corrected light curves</h2>

Kepler took an image of each star every 29.4 minutes. The **simple aperture photometry (SAP)** flux is the number of electrons per second collected in a set of pixels (the "aperture") around the star. This is shown in the top panel. In addition to the stellar light, the SAP photometry also contains:
- slow drifts as the telescope's pointing, focus and temperature change;
- jumps every three months: Kepler rolled by 90 degrees every 'quarter' (~3 months) to keep its solar panels facing the Sun, so each star fell on a different detector, with a different aperture (different quarters are shown in different colours);
- light from neighbouring stars that falls in the aperture, while part of the target's own light falls outside it.

The mission's **Presearch Data Conditioning (PDC)** pipeline corrected for most of this quarter-by-quarter (Smith et al. 2012; Stumpe et al. 2012, 2014). Many stars on the same part of the detector show the same instrumental trends, so PDC fits a set of these common trends ("cotrending basis vectors") to each star and removes them. It also subtracts the light of neighbouring stars and corrects for the target's light outside the aperture (the keywords CROWDSAP and FLFRCSAP), so that transit depths are still correct. The result is the **PDCSAP** flux (bottom panel), which we use from here on. Here each quarter is also divided by its median, which removes the remaining jumps between quarters and makes the flux relative (i.e. 1 = median star brightness).

In [ ]:
def load(change=None):
    busy(out_load, 'Loading the light curves...')
    name = f'Kepler-{w_number.value}' if w_target.value == OTHER else w_target.value
    notes = []
    try:
        if w_target.value == OTHER:
            data = kt.download_light_curves(name)
            ld = (0.4, 0.26)                         # typical values for a Sun-like star in the Kepler band
            try:
                arch = kt.archive_parameters(f'{name} b')
            except Exception as error:
                arch = None
                notes.append(f'Could not get the published parameters of {name} b from the NASA Exoplanet Archive ({error}).')
            if arch is None and not notes:
                notes.append(f'{name} b is not in the NASA Exoplanet Archive.')
        else:
            data = kt.bundled_light_curves(name)
            arch = targets.loc[name]
            ld = (float(arch['u1']), float(arch['u2']))
        S.clear()
        S.update(name=name, data=data, arch=arch, ld=ld)
        draw_raw()
        for n in notes:
            out_load.append_display_data(HTML(f'<p><i>Note: {escape(n)}</i></p>'))
    except Exception as error:
        show_in(out_load, HTML(f'<p style="color:red">Could not load {escape(name)}: {escape(str(error))}</p>'))
        return
    run_from('flatten')

def draw_raw():
    d = S['data']
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    plt.close(fig)
    for q in np.unique(d['quarter']):
        s = d['quarter'] == q
        ax1.plot(d['time'][s], d['sap_raw'][s], '.', ms=1)
    ax2.plot(d['time'], d['pdc'], '.', ms=1, color='k')
    ax1.set_ylabel('SAP flux (e$^-$/s)')
    ax2.set_ylabel('PDCSAP flux (relative)')
    ax2.set_xlabel('Time (BJD - 2454833, days)')
    ax1.set_title(f"{S['name']}: {len(np.unique(d['quarter']))} quarters, {len(d['time'])} measurements")
    fig.tight_layout()
    show_in(out_load, fig)


def choose_target(change):
    other_box.layout.display = 'flex' if w_target.value == OTHER else 'none'
    if w_target.value != OTHER:
        load()


w_target.observe(choose_target, names='value')
w_download.on_click(load)

<h2>2. Flattening</h2>

The PDCSAP light curve can still vary slowly, e.g. stars change in brightness as starspots rotate in and out of view, and some instrumental effects remain. Transits are short (hours), so we remove these slow variations by dividing the light curve by a smooth trend, here calculated with a Savitzky-Golay filter (essentially a running polynomial fit; the details are not important). The window of the filter must be several times longer than a transit, otherwise the filter would follow the transits and make them shallower. The red line in the top panel is the trend; the bottom panel shows the flattened light curve.

We also measure the **scatter** (noise) of the flattened light curve, separately for each quarter: Kepler rotated by 90 degrees every quarter, so the star's light fell on a different detector, and the noise level changes from quarter to quarter. The scatter is used to weight the measurements in the search and in the fit.

In [ ]:
w_window = widgets.FloatSlider(value=2.0, min=0.5, max=6.0, step=0.25, description='Filter window (days):',
                               continuous_update=False, readout_format='.2f', style=style)
out_flat = widgets.Output()
display(w_window, out_flat)


def flatten():
    d = S['data']
    S['flat0'], trend = kt.flatten(d['time'], d['pdc'], w_window.value)
    S['sigma0'] = kt.quarter_sigma(S['flat0'], d['quarter'])      # scatter of each point's quarter
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    plt.close(fig)
    ax1.plot(d['time'], d['pdc'], '.', ms=1, color='k')
    ax1.plot(d['time'], trend, '-', color='red', lw=1)
    ax2.plot(d['time'], S['flat0'], '.', ms=1, color='k')
    ax1.set_ylabel('PDCSAP flux')
    ax2.set_ylabel('Flattened flux')
    ax2.set_xlabel('Time (BJD - 2454833, days)')
    ppm = [S['sigma0'].min()*1e6, np.median(S['sigma0'])*1e6, S['sigma0'].max()*1e6]
    ax2.set_title('Scatter of the flattened light curve: %.0f ppm (parts per million); %.0f to %.0f ppm in different quarters'
                  % (ppm[1], ppm[0], ppm[2]))
    fig.tight_layout()
    show_in(out_flat, fig)


w_window.observe(lambda change: run_from('flatten'), names='value')

<h2>3. Searching for transits: the BLS periodogram</h2>

A transiting planet makes the star slightly fainter for a few hours, once every orbit. The **Box Least Squares** (BLS) method (Kovács et al. 2002) searches for flux drops over many trial periods; for each period, it folds the light curve at that period and fits a box-shaped dip, of several trial durations. The signal-to-noise ratio of the best box at each period gives the **periodogram**: a planet shows up as a peak at its orbital period (as well as smaller peaks at multiples and fractions of it). The **signal detection efficiency** (SDE) tells how far the highest peak stands out above the rest of the periodogram; here the highest peak is taken to be the planet, and in step 4 further peaks count as real if their SDE is above 7.

Searching four years of data on a fine enough grid of periods would take a long time, so the search uses the first few quarters (slider). The period found is then **refined with all the quarters**: over four years a planet orbits hundreds of times, so even a tiny error in the period adds up to a large error in the predicted transit times. The period from this step is therefore precise to a few seconds, but it will be refined further by the fit in step 6.

Right: light curve folded at the best period (time is measured from the middle of the transit).

In [ ]:
w_quarters = widgets.IntSlider(value=4, min=1, max=18, description='Quarters searched:', continuous_update=False,
                               style=style)
w_prange = widgets.FloatRangeSlider(value=[0.5, 20], min=0.3, max=60, step=0.1, description='Periods (days):',
                                    continuous_update=False, readout_format='.1f', style=style,
                                    layout=widgets.Layout(width='500px'))
out_bls = widgets.Output()
display(w_quarters, w_prange, out_bls)


def search():
    d, f = S['data'], S['flat0']
    quarters = np.unique(d['quarter'])
    use = np.isin(d['quarter'], quarters[:w_quarters.value])
    b = kt.bls_search(d['time'][use], f[use], S['sigma0'][use], *w_prange.value)
    S['bls'] = b = kt.refine_period(d['time'], f, S['sigma0'], b)
    hours = 24*kt.phase_time(d['time'], b['t0'], b['period'])
    near = np.abs(hours) < 72*b['duration']
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw=dict(width_ratios=[1.4, 1]))
    plt.close(fig)
    ax1.plot(b['periods'], b['power'], lw=0.6, color='k')
    ax1.axvline(b['period'], color='red', lw=1, alpha=0.4)
    ax1.set_xscale('log')
    ax1.set_xlabel('Trial period (days)')
    ax1.set_ylabel('BLS signal-to-noise')
    ax1.set_title(f"Best period {b['period']:.5f} days (SDE {b['sde']:.1f})")
    ax2.plot(hours[near], f[near], '.', ms=1, color='k')
    ax2.set_xlabel('Time from mid-transit (hours)')
    ax2.set_ylabel('Flattened flux')
    ax2.set_title(f"Depth {b['depth']*1e6:.0f} ppm, duration {b['duration']*24:.1f} h")
    fig.tight_layout()
    show_in(out_bls, fig)


for w in (w_quarters, w_prange):
    w.observe(lambda change: run_from('search'), names='value')

<h2>4. Masking the transits, searching again, and removing outliers</h2>

Now that we know where the transits are, we **mask** them (orange points in the bottom panel) and flatten the light curve again, so that the filter is not pulled down by the transits.

With the transits of the first planet masked, we can **search again** for other signals (top panel; again using the first few quarters, with trial periods up to at least 60 days). Other planets in the same system may show up, but so may other things: for example, a planet passing *behind* its star (an occultation or secondary eclipse, half an orbit after the transit) makes a much shallower dip at the same period. Signals with SDE above 7 are masked as well.

Finally we remove **outliers**, single measurements that are far off (cosmic-ray hits, instrumental glitches): points more than 7 times the scatter (of their quarter) away from the running median of their nearest neighbours (red crosses). A transit comprises a consistent drop in flux (i.e. several contiguous low flux points), so the running median follows it and transits are not removed; the masked transits are protected in any case.

In [ ]:
w_second = widgets.Checkbox(value=True, description='Search for a second signal', style=style)
w_nsig = widgets.FloatSlider(value=7, min=3, max=15, step=0.5, description='Outlier threshold (sigma):',
                             continuous_update=False, readout_format='.1f', style=style)
out_clean = widgets.Output()
display(w_second, w_nsig, out_clean)


def clean():
    d, b = S['data'], S['bls']
    t = d['time']
    mask = kt.transit_mask(t, b['period'], b['t0'], b['duration'])
    flat, _ = kt.flatten(t, d['pdc'], w_window.value, mask=mask)
    sigma = kt.quarter_sigma(flat, d['quarter'], ~mask)       # scatter outside the transits, per quarter
    second, note = None, 'Second search switched off.'
    other = np.zeros(len(t), bool)                            # transits of a second signal
    if w_second.value:
        quarters = np.unique(d['quarter'])
        use = np.isin(d['quarter'], quarters[:w_quarters.value]) & ~mask
        second = kt.bls_search(t[use], flat[use], sigma[use], w_prange.value[0], max(w_prange.value[1], 60))
        if second['sde'] >= 7:
            second = kt.refine_period(t[~mask], flat[~mask], sigma[~mask], second)
            other = kt.transit_mask(t, second['period'], second['t0'], second['duration'])
            note = f"Second signal found: period {second['period']:.4f} d, depth {second['depth']*1e6:.0f} ppm, SDE {second['sde']:.1f} (masked)."
        else:
            note = f"Highest peak: period {second['period']:.3f} d, depth {second['depth']*1e6:.0f} ppm, SDE {second['sde']:.1f}: not significant."
        # same period as the first planet (or half of it), half an orbit after its transits?
        ratio = second['period']/b['period']
        phase = abs(kt.phase_time(second['t0'], b['t0'], b['period'])/b['period'])        # 0 ... 0.5
        if (abs(ratio - 1) < 2e-3 and phase > 0.45) or (abs(ratio - 0.5) < 1e-3 and min(phase, 0.5 - phase) < 0.05):
            note += (' This is at the period of the first planet (or half of it), half an orbit after its transits: '
                     + ('probably' if second['sde'] >= 7 else 'possibly') + ' the planet passing behind its star (an occultation).')
    mask = mask | other
    out = kt.find_outliers(flat, sigma, mask, w_nsig.value)
    S.update(flat=flat, sigma=sigma, mask=mask, other=other, keep=~out, second=second)

    fig = plt.figure(figsize=(10, 6.5))
    plt.close(fig)
    if second is not None:
        ax1 = fig.add_subplot(2, 1, 1)
        ax1.plot(second['periods'], second['power'], lw=0.6, color='k')
        ax1.set_xscale('log')
        ax1.set_xlabel('Trial period (days)')
        ax1.set_ylabel('BLS signal-to-noise')
        ax1.set_title('Second search (first planet masked)')
    ax2 = fig.add_subplot(2, 1, 2)
    ax2.plot(t[mask], flat[mask], '.', ms=1, color='tab:orange')
    ax2.plot(t[~mask], flat[~mask], '.', ms=1, color='k')
    ax2.plot(t[out], flat[out], 'x', color='red', ms=5)
    ax2.set_xlabel('Time (BJD - 2454833, days)')
    ax2.set_ylabel('Flattened flux')
    ax2.set_title(f'{out.sum()} outliers removed (red crosses); masked transits in orange; scatter {np.median(sigma)*1e6:.0f} ppm')
    fig.tight_layout()
    show_in(out_clean, HTML(f'<p>{note}</p>'))
    out_clean.append_display_data(fig)


for w in (w_second, w_nsig):
    w.observe(lambda change: run_from('clean'), names='value')

<h2>5. Modelling the transit by hand</h2>

A transit light curve depends on (Mandel & Agol 2002; here computed with the `batman` package, Kreidberg 2015):
- the time of mid-transit $t_0$ (the slider gives the offset from the BLS value, in minutes) and the orbital period $P$ (fixed here at the BLS value);
- the ratio of the planet's and the star's radii $R_{\rm p}/R_*$, which sets the depth (roughly $(R_{\rm p}/R_*)^2$ if ignoring limb darkening);
- the orbital distance in units of the stellar radius $a/R_*$ and the inclination $i$, which together set the duration and the shape. The impact parameter $b = (a/R_*)\cos i$ (for a circular orbit) is the closest distance between the planet and the centre of the star, projected on the sky, in stellar radii. For $a/R_* \gg 1$ the total duration of the transit is $T_{\mathrm{dur}} \approx (P/\pi)\,(R_*/a)\sqrt{(1 + R_{\rm p}/R_*)^2 - b^2}$;
- **limb darkening**: a star looks darker towards its edge, which rounds off the bottom of the transit. We describe it with two coefficients $u_1, u_2$ (quadratic law), here fixed at values calculated for the star's temperature, surface gravity and metallicity (Claret & Bloemen 2011). Typical values for a Sun-like star would be $u_1 = 0.4$, $u_2 = 0.26$).

Each measurement is an average over a 29.4-minute exposure, which smooths the short ingress and egress, so the model is averaged in the same way: it is calculated at 21 points in time spread evenly over each exposure and averaged (with too few, the fitted parameters come out biased; Kipping 2010). Try to match the folded data (black: individual measurements; blue: averages in 15-minute bins) by moving the sliders, or type values into the boxes. Below the plot: the data minus the model, i.e. residuals. The title gives the reduced $\chi^2$. For a perfect model and purely random noise it would be close to 1; here some instrumental and stellar effects remain in the data, so even the best model gives about 1.3 to 1.7 for the three pre-selected stars. Use it, together with your visual estimate, to compare models (remember that a lower $\chi^2$ is better).

In [ ]:
w_dt0 = widgets.FloatSlider(value=0, min=-60, max=60, step=0.5, description='t0 offset (min):', continuous_update=False,
                            readout_format='.1f', style=style)
w_rp = widgets.FloatSlider(value=0.1, min=0.002, max=0.3, step=0.0002, description='Rp/R*:', continuous_update=False,
                           readout_format='.4f', style=style)
w_a = widgets.FloatSlider(value=10, min=1.5, max=40, step=0.01, description='a/R*:', continuous_update=False,
                          readout_format='.2f', style=style)
w_inc = widgets.FloatSlider(value=89, min=60, max=90, step=0.05, description='i (degrees):', continuous_update=False,
                            readout_format='.2f', style=style)
w_u1 = widgets.BoundedFloatText(value=0.4, min=-1, max=2, step=0.01, description='u1:', layout=widgets.Layout(width='130px'))
w_u2 = widgets.BoundedFloatText(value=0.26, min=-1, max=2, step=0.01, description='u2:', layout=widgets.Layout(width='130px'))
out_model = widgets.Output()
display(w_dt0, w_rp, w_a, w_inc, widgets.HBox([w_u1, w_u2]), out_model)


def model_sliders_from_bls():
    # Starting guesses from the BLS result: depth -> Rp/R*, duration -> a/R* (for a central transit)
    b = S['bls']
    S['quiet'] = True                  # don't redraw for every slider we set here
    try:
        w_dt0.value = 0
        w_rp.value = float(np.clip(np.sqrt(max(b['depth'], 1e-6)), w_rp.min, w_rp.max))
        w_a.value = float(np.clip(b['period']/(np.pi*b['duration']), w_a.min, w_a.max))
        w_inc.value = 89.5
        w_u1.value, w_u2.value = S['ld']
    finally:
        S['quiet'] = False


def folded_data():
    # Data within two transit durations of mid-transit; outliers and transits of a second signal removed
    d, b = S['data'], S['bls']
    keep = S['keep'] & ~S['other'] & (np.abs(kt.phase_time(d['time'], b['t0'], b['period'])) < 2*b['duration'])
    return d['time'][keep], S['flat'][keep], S['sigma'][keep]


def draw_fold(out, t, f, s, models, title, nfree=4):
    last = models[-1][1]                       # fold the data with the last model's period and t0
    hours = 24*kt.phase_time(t, last['t0'], last['period'])
    grid = np.linspace(hours.min(), hours.max(), 1000)
    bins = np.arange(hours.min(), hours.max() + 0.25, 0.25)
    idx = np.digitize(hours, bins)
    count = np.bincount(idx)
    full = [k for k in np.unique(idx) if count[k] >= 0.5*np.median(count[count > 0])]    # skip part-filled edge bins
    centres = [hours[idx == k].mean() for k in full]
    means = [f[idx == k].mean() for k in full]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True, gridspec_kw=dict(height_ratios=[3, 1]))
    plt.close(fig)
    ax1.plot(hours, f, '.', ms=1, color='k', alpha=0.4)
    ax1.plot(centres, means, 'o', ms=4, color='tab:blue')
    for label, p, style_ in models:            # each model drawn around its own mid-transit time
        m = kt.transit_model(p['t0'] + grid/24, p['t0'], p['period'], p['rp_rs'], p['a_rs'], p['inc'], p['u1'], p['u2'])
        ax1.plot(grid, m, style_, lw=2, label=label)
    resid = f - kt.transit_model(t, last['t0'], last['period'], last['rp_rs'], last['a_rs'], last['inc'], last['u1'], last['u2'])
    rbin = [resid[idx == k].mean() for k in full]
    ax2.plot(hours, resid*1e6, '.', ms=1, color='k', alpha=0.4)
    ax2.plot(centres, np.array(rbin)*1e6, 'o', ms=4, color='tab:blue')
    ax2.axhline(0, color='red', lw=1)
    lo, hi = np.percentile(f, [0.5, 99.5])
    s_typ = np.median(s)
    ax1.set_ylim(min(lo, min(means)) - 2*s_typ, hi + s_typ)
    ax2.set_ylim(-5*s_typ*1e6, 5*s_typ*1e6)
    ax1.set_ylabel('Flattened flux')
    ax2.set_ylabel('Data - model (ppm)')
    ax2.set_xlabel('Time from mid-transit (hours)')
    ax1.legend(loc='lower right')
    chi2 = np.sum((resid/s)**2)/(len(f) - nfree)
    ax1.set_title(f'{title}; reduced $\\chi^2$ = {chi2:.2f}')
    fig.tight_layout()
    show_in(out, fig)


def manual_model(change=None):
    if S.get('quiet') or 'keep' not in S:     # not ready (or an earlier step failed)
        return
    try:
        b = S['bls']
        p = dict(t0=b['t0'] + w_dt0.value/1440, period=b['period'], rp_rs=w_rp.value, a_rs=w_a.value, inc=w_inc.value,
                 u1=w_u1.value, u2=w_u2.value)
        t, f, s = folded_data()
        b_imp = kt.impact_parameter(w_a.value, w_inc.value)
        draw_fold(out_model, t, f, s, [('your model', p, 'r-')],
                  f"{S['name']}: period {b['period']:.5f} d, impact parameter b = {b_imp:.2f}")
    except Exception as error:
        show_in(out_model, HTML(f'<p style="color:red">{escape(str(error))}</p>'))


for w in (w_dt0, w_rp, w_a, w_inc, w_u1, w_u2):
    w.observe(manual_model, names='value')

<h2>6. Fitting the transit model</h2>

Instead of adjusting the parameters by hand, we can find the 'best-fitting' parameters automatically by minimising $\chi^2$ (the sum of the squared differences between data and model, each divided by the square of the measurement scatter). This is known as a **least-squares fit**. The fit starts from the values published in the NASA Exoplanet Archive, deliberately shifted from their true values so you can see that the fit finds its way back. With four years of data the fit also refines the period.

Tick the parameters to fit; unticked parameters stay at their starting values. Limb darkening can be fitted as well, but do this last (if selected, we use the Kipping (2013) parameters $q_1, q_2$ instead of $u_1, u_2$ to keep the star's brightness profile physical (brightest at the centre) as long as $0 \le q_1, q_2 \le 1$. Fitting limb darkening needs a deep, well-measured transit: it works for the two hot Jupiters (Keplers 7b and 8b), but for the much smaller Kepler-10 b the data cannot tell limb darkening apart from the orbit's geometry, and the fit may fail.

The table compares the fit with published values (for the pre-selected stars: Esteves et al. 2015). With limb darkening fixed at the calculated values, $a/R_*$ and $i$ of the two hot Jupiters come out many uncertainties away from the published ones: the calculated limb darkening does not quite match these stars, and the fit compensates by changing the shape of the transit through $a/R_*$ and $i$. With limb darkening fitted as well, the results agree with the published values to within a few $\sigma$. Some difference is expected: published analyses used other data (e.g. short-cadence measurements) and methods, and the uncertainties of a simple least-squares fit are too small when the noise is not purely random (reduced $\chi^2$ above 1).

Parameters marked "at limit" hit the edge of their allowed range (e.g. $i = 90°$). They have no meaningful uncertainty, and the uncertainties of the other parameters are then calculated as if the parameter at its limit were fixed, so they will be too small.

Some parameters can compensate for each other. For a small planet whose transit crosses close to the centre of the star (e.g. Kepler-10 b), the light curve mainly tells us the depth and the duration of the transit; the short ingress and egress are smeared out by the 29.4-minute exposures. A larger $a/R_*$ (which shortens the transit) can then be compensated by a smaller impact parameter $b$, i.e. an inclination closer to 90° (the planet then crosses a longer chord of the star, i.e. closer to the centre of the star), and the fit cannot constrain $a/R_*$ and $i$ separately: the table reports them as strongly correlated, or "not constrained by the data". Published inclinations of Kepler-10 b likewise range from about 84° (with uncertainties of several degrees) to almost 90°. Try unticking the inclination (it will stay at its starting value) and fitting with different starting shifts, e.g. 0 and 1 $\sigma$: $a/R_*$ changes, but the reduced $\chi^2$ hardly does.

In [ ]:
w_shift = widgets.BoundedFloatText(value=3, min=0, max=20, step=0.5, description='Start shifted by (sigma):',
                                   style=style, layout=widgets.Layout(width='250px'))
w_free = {n: widgets.Checkbox(value=(n != 'ld'), description=label, indent=False, layout=widgets.Layout(width='150px'))
          for n, label in [('t0', 't0'), ('period', 'period'), ('rp_rs', 'Rp/R*'), ('a_rs', 'a/R*'), ('inc', 'inclination'),
                           ('ld', 'limb darkening')]}
w_fit = widgets.Button(description='Fit')
out_fit = widgets.Output()
display(w_shift, widgets.HBox(list(w_free.values())), w_fit, out_fit)


def fit(button=None):
    if 'keep' not in S:
        show_in(out_fit, HTML('<p style="color:red">The steps above have not been completed.</p>'))
        return
    busy(out_fit, 'Fitting...')
    try:
        A, b = S['arch'], S['bls']
        t, f, s = folded_data()
        t_mid = b['t0'] + np.round((np.median(t) - b['t0'])/b['period'])*b['period']   # a transit near the middle of the data
        notes = []
        if A is None or not np.all(np.isfinite([A['rp_rs'], A['a_rs'], A['inc'], A['t0_bjd'], A['period']])):
            A = None
            notes.append('no complete set of published transit parameters for this planet: the fit started from your values in step 5.')
        elif abs(A['period']/b['period'] - 1) > 0.01:
            notes.append(f"the BLS period ({b['period']:.4f} d) is not the published period of {A.get('planet', 'planet b')} "
                         f"({A['period']:.4f} d): the search found a different signal, so the fit started from your values "
                         "in step 5 and there are no published values to compare with.")
            A = None
        if A is None:
            start = dict(t0=t_mid + w_dt0.value/1440, period=b['period'], rp_rs=w_rp.value, a_rs=w_a.value, inc=w_inc.value)
        else:
            start, more = kt.start_from_archive(A, t_mid, w_shift.value)
            notes += more
        free = {n: w.value for n, w in w_free.items()}
        r = kt.fit_transit(t, f, s, start, free, w_u1.value, w_u2.value, b['period'])
        S['fit'] = r
        draw_fold(out_fit, t, f, s, [('start', r['start'], 'g--'), ('fit', r['best'], 'r-')],
                  f"{S['name']}: least-squares fit to {r['n_points']} points", nfree=len(r['free']))
        out_fit.append_display_data(HTML(results_table(r, A, notes + r['notes'])))
    except Exception as error:
        show_in(out_fit, HTML(f'<p style="color:red">The fit failed: {escape(str(error))}</p>'))

def results_table(r, A, notes):
    B, E, start = r['best'], r['errors'], r['start']
    short = dict(t0='t0', period='period', rp_rs='Rp/R*', a_rs='a/R*', inc='i', q1='q1', q2='q2')
    rows = [('t0 (BJD - 2454833)', 't0', 1, '.5f'), ('period (days)', 'period', 1, '.7f'), ('Rp/R*', 'rp_rs', 1, '.5f'),
            ('a/R*', 'a_rs', 1, '.3f'), ('inclination (deg)', 'inc', 1, '.2f'), ('u1', 'u1', 1, '.3f'), ('u2', 'u2', 1, '.3f')]
    html = ['<table><tr><th>parameter</th><th>published</th><th>start</th><th>fit</th><th></th></tr>']
    published = {}
    if A is not None:                  # (value, uncertainty); the epoch is moved to the fitted one
        published = dict(t0=kt.archive_epoch(A, start['t0']), period=(A['period'], A['period_err']),
                         rp_rs=(A['rp_rs'], A['rp_rs_err']), a_rs=(A['a_rs'], A['a_rs_err']), inc=(A['inc'], A['inc_err']))
    for label, n, _, fmt in rows:
        pub, perr = published.get(n, (np.nan, np.nan))
        pub_s = '' if not np.isfinite(pub) else f'{pub:{fmt}}' + (f' ± {perr:{fmt}}' if np.isfinite(perr) else '')
        st = start[n]
        fitted = n in r['free'] or (n in ('u1', 'u2') and 'q1' in r['free'])
        q = {'q1', 'q2'} if n in ('u1', 'u2') else {n}
        if not fitted:
            fit_s = f'{B[n]:{fmt}} (fixed)'
        elif n in ('u1', 'u2'):
            fit_s = f'{B[n]:{fmt}}'                     # uncertainties are those of q1, q2 (see below)
        else:
            fit_s = f'{B[n]:{fmt}}' + (f' ± {E[n]:{fmt}}' if n in E and n not in r['unconstrained'] else '')
        flag = ('at limit' if q & set(r['at_limit']) else
                'not constrained by the data' if q & set(r['unconstrained']) else '')
        html.append(f'<tr><td>{label}</td><td>{pub_s}</td><td>{st:{fmt}}</td><td>{fit_s}</td><td>{flag}</td></tr>')
    html.append('</table>')
    ref = '' if A is None else f" Published values: {A['reference']} (NASA Exoplanet Archive)."
    html.append(f"<p>Reduced chi-squared {r['chi2_red']:.2f}; impact parameter b = "
                f"{kt.impact_parameter(B['a_rs'], B['inc']):.2f}.{ref}</p>")
    if 'q1' in r['free']:
        qs = [f"{n} = {B[n]:.3f}" + (f" ± {E[n]:.3f}" if n in E and n not in r['unconstrained'] else '') for n in ('q1', 'q2')]
        html.append(f"<p>Limb darkening fitted as {', '.join(qs)} (Kipping 2013).</p>")
    if r['correlated']:
        pairs = ', '.join(f'{short[a]} and {short[b]} ({c:+.2f})' for a, b, c in r['correlated'])
        html.append(f'<p><i>Strongly correlated: {pairs}. The data constrain mainly a combination of these parameters; '
                    'their uncertainties (calculated as if the model were linear) can then be unreliable.</i></p>')
    html += [f'<p><i>Note: {escape(n)}</i></p>' for n in notes]
    return '\n'.join(html)


w_fit.on_click(fit)

In [ ]:
# Run the analysis: changing something in one step recalculates all later steps
STEPS = ['flatten', 'search', 'clean', 'model']


PRODUCES = dict(flatten=['flat0', 'sigma0'], search=['bls'], clean=['flat', 'sigma', 'mask', 'other', 'keep', 'second'],
                model=[])                  # results stored in S by each step


def run_from(step):
    outs = dict(flatten=out_flat, search=out_bls, clean=out_clean, model=out_model)
    todo = STEPS[STEPS.index(step):]
    for s in todo:
        busy(outs[s])
        for key in PRODUCES[s]:
            S.pop(key, None)
    S.pop('fit', None)
    show_in(out_fit, HTML('<p>Press <b>Fit</b> to fit the transit model.</p>'))
    for i, s in enumerate(todo):
        try:
            if s == 'flatten':
                flatten()
            elif s == 'search':
                search()
            elif s == 'clean':
                clean()
            elif s == 'model':
                model_sliders_from_bls()
                manual_model()
        except Exception as error:      # show the problem in this step and stop
            show_in(outs[s], HTML(f'<p style="color:red">This step failed: {escape(str(error))}</p>'))
            for s2 in todo[i + 1:]:
                show_in(outs[s2], HTML('<p><i>Not calculated, because an earlier step failed.</i></p>'))
            return

load()

<h3>References</h3>

Astropy Collaboration 2022, ApJ 935, 167 (astropy, whose BLS implementation we are using) $\cdot$ Claret & Bloemen 2011, A&A 529, A75 (limb darkening) $\cdot$ Esteves et al. 2015, ApJ 804, 150 (published parameters of the the pre-selected planets) $\cdot$ Kipping 2010, MNRAS 408, 1758 (exposure-time smearing) $\cdot$ Kipping 2013, MNRAS 435, 2152 ($q_1, q_2$) $\cdot$ Kovács, Zucker & Mazeh 2002, A&A 391, 369 (BLS) $\cdot$ Kreidberg 2015, PASP 127, 1161 (batman) $\cdot$ Lightkurve Collaboration 2018, Astrophysics Source Code Library, ascl:1812.013 (lightkurve) $\cdot$ Mandel & Agol 2002, ApJ 580, L171 (transit model) $\cdot$ Smith et al. 2012, PASP 124, 1000; Stumpe et al. 2012, PASP 124, 985; Stumpe et al. 2014, PASP 126, 100 (Kepler PDC).